In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
df = pd.read_csv('sales_data.csv')

In [ ]:
df

## 1. Basic Checks

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.describe(include='O')

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.nunique()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

#### Findings:
- Dataset has 510 rows and 10 columns
- Columns: order_id, date, product, category, region, salesperson, quantity, unit_price, discount, total_sales
- Mixed data types — numeric and text columns present
- Duplicate rows detected
- Missing values found in region, salesperson, discount and total_sales

### 2. Missing Values

In [ ]:
df.isnull().sum()

#### Missing values found in:
- region (~23 missing)
- salesperson (~27 missing)
- discount (~27 missing)
- total_sales (~10 missing)

region and salesperson are categorical — missing values will be replaced with mode (most frequent value)

discount and total_sales are numerical — missing values will be replaced with median

### 3. Checking Inconsistent Text

In [ ]:
df['product'].unique()

In [ ]:
df['region'].unique()

In [ ]:
df['salesperson'].unique()

In [ ]:
df['category'].unique()

- region column has inconsistent casing — 'north','NORTH','SOUTH','south' 'east','EAST', 'WEST','west' all refer to the same values
- All text columns will be standardized to small Case and extra whitespace will be removed
- Note: after applying str.title(), any NaN values become the string 'Nan' — these will be converted back to actual NaN before filling

### 4. Checking Invalid Numeric Values

In [ ]:
# checking what unit_price actually looks like — some entries may have text in them
df['unit_price'].unique()

Some unit_price entries contain text like '55000 INR' instead of a plain number. Non-numeric characters will be stripped and the column will be converted to float.

In [ ]:
# checking for negative or zero total_sales — these are not valid business entries
df[df['total_sales'] <= 0]

In [ ]:
df[df['quantity'] <= 0]

- Rows with total_sales <= 0 are invalid — a sale cannot be zero or negative
- These rows will be dropped as they have no business meaning

#### Discount

In [ ]:
# checking discount values — some may have been entered as 15 instead of 0.15
df['discount'].unique()

Discount should always be between 0 and 1 (e.g. 0.10 means 10%). Any value greater than 1 was entered as a percentage (e.g. 15) and will be divided by 100.

### 5. Outlier Check

In [ ]:
# visualizing distribution of total_sales to check for extreme values
sns.boxplot(x=df['total_sales'])

The boxplot shows some high total_sales values. However these are not outliers — a large Laptop order at full price is a valid high-value sale. Removing valid high sales using IQR would reduce our total_sales and skew the top_product result.

The only invalid entries are negative or zero values which will be dropped in preprocessing.

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np


def preprocess(filepath):

    df = pd.read_csv(filepath)

    # removing duplicate rows
    df = df.drop_duplicates()

    # converting text columns values to title case and removing extra spaces
    for col in ['product', 'region', 'salesperson', 'category']:
        df[col] = df[col].astype(str).str.strip().str.title()

    # after applying str.title(), NaN becomes the string "Nan"
    # converting it back to actual NaN so we can fill it properly
    df[['region', 'salesperson']] = df[['region', 'salesperson']].replace('Nan', np.nan)

    # filling missing categorical values with the most frequent value
    for col in ['region', 'salesperson']:
        df[col] = df[col].fillna(df[col].mode()[0])

    # cleaning unit_price first before filling nulls
    # some entries have text like "55000 INR" so we strip non-numeric characters
    df['unit_price'] = df['unit_price'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
    df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
    df['unit_price'] = df['unit_price'].fillna(df['unit_price'].median())

    # converting quantity and discount to numeric
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
    df['discount'] = pd.to_numeric(df['discount'], errors='coerce')
    df['discount'] = df['discount'].fillna(df['discount'].median())

    # some discount values were entered as 15 instead of 0.15
    # dividing by 100 to bring them to the correct decimal format
    df.loc[df['discount'] > 1, 'discount'] /= 100

    # removing rows where quantity is zero or negative as they are not valid orders
    df = df[df['quantity'] > 0]

    # converting total_sales to numeric and removing negative or zero values
    # negative sales have no business meaning so we drop them
    df['total_sales'] = pd.to_numeric(df['total_sales'], errors='coerce')
    df = df[df['total_sales'] > 0]
    df['total_sales'] = df['total_sales'].fillna(df['total_sales'].median())

    # converting the date column to datetime format
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date'])

    # keeping only the columns needed for analysis
    df = df[['date', 'product', 'region', 'total_sales']]

    # dropping duplicates again after column selection
    # two rows might look different with all columns but become identical once we trim down to 4
    df = df.drop_duplicates()

    df = df.reset_index(drop=True)

    return df

if __name__ == "__main__":
    df = preprocess("ml/TeamA/data/sales_data.csv")
    print(df.head())

In [ ]:
df = preprocess("ml/TeamA/data/sales_data.csv")

print("Shape:", df.shape)
print("Missing values:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print("Negative total_sales:", (df["total_sales"] <= 0).sum())

In [ ]:
df

After preprocessing:
- All duplicates removed
- No missing values remaining
- All text columns standardized to Title Case
- Invalid rows (negative/zero total_sales, zero quantity) dropped
- Only relevant columns retained: date, product, region, total_sales